# Active Learning em texto: classificando resenhas de filmes (Keras 3)

**Cenário realista:** você tem 10.000 resenhas de filmes e quer um classificador de sentimento (positivo/negativo),
mas **cada rótulo custa tempo de uma pessoa** lendo a resenha. Quais resenhas vale a pena rotular primeiro?

O active learning responde isso em ciclos:

1. Começa com poucas resenhas rotuladas (200)
2. Treina um modelo
3. O modelo aponta as resenhas em que **mais tem dúvida**
4. Um "oráculo" (aqui simulado; no final há um modo em que **você** rotula) informa os rótulos delas
5. Retreina com as novas e repete

Comparamos 4 formas de escolher o que rotular:

| Estratégia | Ideia |
|---|---|
| `random` | baseline: sorteia (é o que acontece quando não há critério) |
| `least_confidence` | escolhe as resenhas cuja previsão está mais perto de 50% |
| `entropy` | escolhe onde a distribuição prevista é mais incerta |
| `bald` | usa **MC Dropout**: várias previsões com dropout ligado; escolhe onde o modelo "discorda de si mesmo" (incerteza do modelo, não só do dado) |

No fim você também vê **quais resenhas** o modelo pediu para rotular.

In [ ]:
# Se necessário, descomente para instalar:
# !pip install -q "keras>=3" tensorflow matplotlib numpy

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # pode trocar por "jax" ou "torch"

import time
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

print("Keras:", keras.__version__, "| backend:", keras.backend.backend())

## 1. Dados: resenhas do IMDB

O Keras já traz o IMDB (50.000 resenhas, rótulo 1 = positiva, 0 = negativa), com cada palavra convertida em um número.
Usamos:
- **Pool** (10.000 resenhas): o conjunto "sem rótulo" de onde o modelo pode pedir exemplos
- **Teste** (10.000 resenhas): fixo, só para medir a acurácia

Cada resenha é cortada/preenchida para 200 palavras.

In [ ]:
VOCAB, MAXLEN, POOL_SIZE = 10_000, 200, 10_000

(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
pad = lambda seqs: keras.utils.pad_sequences(seqs, maxlen=MAXLEN, padding="post", truncating="post")

rng0 = np.random.default_rng(0)
pool_idx = rng0.permutation(len(x_tr))[:POOL_SIZE]
test_idx = rng0.permutation(len(x_te))[:10_000]

x_pool_raw = [x_tr[i] for i in pool_idx]          # versão original (para decodificar em texto)
x_pool, y_pool = pad(x_pool_raw), y_tr[pool_idx]
x_test, y_test = pad([x_te[i] for i in test_idx]), y_te[test_idx]

print("Pool:", x_pool.shape, "| Teste:", x_test.shape, "| % positivas no pool:", y_pool.mean().round(3))

In [ ]:
# Para ler as resenhas, convertemos os números de volta em palavras
word_index = keras.datasets.imdb.get_word_index()
index_word = {v + 3: k for k, v in word_index.items()}   # o Keras reserva os índices 0-3

def decode(seq):
    return " ".join(index_word.get(i, "?") for i in seq if i > 3)

for i in range(3):
    print(f"[{'positiva' if y_pool[i] else 'negativa'}] {decode(x_pool_raw[i])[:250]}...\n")

## 2. Modelo

Uma CNN 1D pequena sobre *embeddings* de palavras, com `Dropout` em várias camadas.
Esses dropouts têm um segundo papel: depois serão usados no **MC Dropout** (estratégia `bald`).

In [ ]:
def build_model():
    model = keras.Sequential([
        keras.Input(shape=(MAXLEN,)),
        layers.Embedding(VOCAB, 32),
        layers.Dropout(0.3),
        layers.Conv1D(64, 5, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

build_model().summary()

## 3. Estratégias de consulta

Cada função devolve um **score de incerteza** (maior = mais valioso rotular). Como a tarefa é binária,
`p` é a probabilidade prevista de a resenha ser positiva.

No **BALD**, fazemos `T` previsões com o dropout ligado. Se todas concordam, o modelo está seguro. Se elas
divergem entre si, ele está incerto sobre *o que aprender*, e rotular aquela amostra ensina mais.
O score é: *entropia da média das previsões − média das entropias*.

In [ ]:
def _entropy(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def score_least_confidence(p):
    return 1.0 - np.maximum(p, 1.0 - p)

def score_entropy(p):
    return _entropy(p)

def score_bald(p_samples):
    # p_samples: shape (T, n) -> T previsões estocásticas para n amostras
    return _entropy(p_samples.mean(axis=0)) - _entropy(p_samples).mean(axis=0)


def mc_predict(model, x, T=10, batch_size=1024):
    """T passagens com dropout LIGADO (training=True). Retorna (T, n)."""
    outs = []
    for _ in range(T):
        parts = [
            keras.ops.convert_to_numpy(model(x[i:i + batch_size], training=True))[:, 0]
            for i in range(0, len(x), batch_size)
        ]
        outs.append(np.concatenate(parts))
    return np.stack(outs)


def query(strategy, model, x_unlabeled, k, rng):
    """Devolve as posições (dentro de x_unlabeled) das k amostras mais valiosas."""
    if strategy == "random":
        scores = rng.random(len(x_unlabeled))
    elif strategy == "bald":
        scores = score_bald(mc_predict(model, x_unlabeled))
    else:
        p = model.predict(x_unlabeled, batch_size=1024, verbose=0)[:, 0]
        scores = {"least_confidence": score_least_confidence, "entropy": score_entropy}[strategy](p)
    return np.argsort(-scores)[:k]

STRATEGIES = ["random", "least_confidence", "entropy", "bald"]

## 4. O laço de active learning

**treinar → avaliar → pontuar o pool → rotular os top-K → repetir.**
Cada ciclo retreina um modelo **novo**, e o mesmo `seed` dá o mesmo conjunto inicial a todas as estratégias, para a comparação ser justa.

In [ ]:
def run_active_learning(strategy, seed, n_init=200, query_size=200, n_rounds=8, epochs=10, verbose=True):
    rng = np.random.default_rng(seed)
    keras.utils.set_random_seed(seed)

    labeled = rng.choice(POOL_SIZE, size=n_init, replace=False).tolist()
    history = []

    for r in range(n_rounds + 1):
        keras.backend.clear_session()
        model = build_model()
        model.fit(x_pool[labeled], y_pool[labeled], epochs=epochs, batch_size=32, verbose=0)

        acc = model.evaluate(x_test, y_test, batch_size=1024, verbose=0)[1]
        history.append((len(labeled), acc))
        if verbose:
            print(f"  [{strategy:16s} seed={seed}] rótulos={len(labeled):5d}  acc={acc:.4f}")
        if r == n_rounds:
            break

        unlabeled = np.setdiff1d(np.arange(POOL_SIZE), labeled)
        pick = query(strategy, model, x_pool[unlabeled], query_size, rng)
        labeled.extend(unlabeled[pick].tolist())          # o oráculo revela os rótulos

    return np.array(history), labeled, model

## 5. Rodando o experimento

4 estratégias × 3 seeds × 9 treinos cada. Em CPU leva alguns minutos; para um teste rápido,
use `SEEDS = [0]` e `N_ROUNDS = 4`.

In [ ]:
SEEDS = [0, 1, 2]
N_ROUNDS = 8           # 200 iniciais + 8 x 200 = 1800 rótulos no final

results = {s: [] for s in STRATEGIES}
final_labeled, last_model = {}, {}

t0 = time.time()
for strategy in STRATEGIES:
    for seed in SEEDS:
        hist, labeled, model = run_active_learning(strategy, seed, n_rounds=N_ROUNDS, verbose=(seed == SEEDS[0]))
        results[strategy].append(hist)
        if seed == SEEDS[0]:
            final_labeled[strategy], last_model[strategy] = labeled, model
print(f"\nTempo total: {(time.time() - t0) / 60:.1f} min")

## 6. Resultado: acurácia × número de rótulos

Linha = média das seeds; faixa = ± 1 desvio-padrão.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for strategy, runs in results.items():
    runs = np.array(runs)
    xs, accs = runs[0, :, 0], runs[:, :, 1]
    mean, std = accs.mean(0), accs.std(0)
    ax.plot(xs, mean, "--" if strategy == "random" else "-", marker="o", ms=4, label=strategy)
    ax.fill_between(xs, mean - std, mean + std, alpha=0.12)

ax.set_xlabel("Número de resenhas rotuladas")
ax.set_ylabel("Acurácia no teste")
ax.set_title("Active learning vs. amostragem aleatória (IMDB)")
ax.grid(alpha=0.3); ax.legend()
plt.show()

In [ ]:
def labels_to_reach(runs, target):
    runs = np.array(runs)
    mean, xs = runs[:, :, 1].mean(0), runs[0, :, 0]
    return int(xs[np.argmax(mean >= target)]) if (mean >= target).any() else None

best_random = np.array(results["random"])[:, -1, 1].mean()
targets = [round(best_random - d, 3) for d in (0.06, 0.03, 0.0)]

for target in targets:
    print(f"\nAcurácia-alvo: {target:.1%}")
    for strategy, runs in results.items():
        n = labels_to_reach(runs, target)
        print(f"  {strategy:16s} -> {'não atingiu' if n is None else f'{n} rótulos'}")

print("\nAcurácia final (média das seeds):")
for strategy, runs in results.items():
    final = np.array(runs)[:, -1, 1]
    print(f"  {strategy:16s} {final.mean():.4f} ± {final.std():.4f}")

## 7. O que o modelo pediu para rotular?

As estratégias ativas não escolhem resenhas "típicas", e sim as **difíceis**. Vamos ver exemplos reais
(estratégia `bald`, seed 0): resenhas com opiniões mistas, sarcasmo ou críticas que elogiam e criticam ao mesmo tempo.

In [ ]:
n_init = 200
queried = final_labeled["bald"][-6:]
p = last_model["bald"].predict(x_pool[queried], verbose=0)[:, 0]

for i, prob in zip(queried, p):
    real = "positiva" if y_pool[i] else "negativa"
    print(f"Rótulo real: {real:8s} | previsão do modelo: {prob:.0%} positiva")
    print(decode(x_pool_raw[i])[:450] + "...\n")

In [ ]:
# Viés das estratégias: elas favorecem resenhas curtas ou longas?
lengths = (x_pool != 0).sum(axis=1)
fig, ax = plt.subplots(figsize=(6, 3.5))
means = {s: lengths[final_labeled[s][n_init:]].mean() for s in STRATEGIES}
ax.bar(means.keys(), means.values())
ax.axhline(lengths.mean(), color="k", ls="--", lw=1, label="média do pool")
ax.set_ylabel("Palavras por resenha (média)")
ax.set_title("Tamanho médio das resenhas consultadas")
ax.legend(); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 8. Modo humano no laço (opcional)

Agora **você** é o oráculo. A função abaixo treina um modelo com 100 resenhas, mostra as `k` em que ele mais
tem dúvida e pede que você digite `1` (positiva) ou `0` (negativa). **Os seus rótulos** é que entram no
treino, e no final ela compara com os rótulos reais, para você ver o quanto um humano também erra nos casos ambíguos.

Para executar, descomente a última linha. (Funciona no Colab ou no Jupyter; as resenhas estão em inglês.)

In [ ]:
def active_learning_com_humano(n_init=100, k=5, n_rounds=3, epochs=10, seed=0):
    rng = np.random.default_rng(seed)
    keras.utils.set_random_seed(seed)
    labeled = rng.choice(POOL_SIZE, size=n_init, replace=False).tolist()
    y_known = {i: int(y_pool[i]) for i in labeled}     # conjunto inicial: rótulos verdadeiros
    human = []

    for r in range(n_rounds):
        keras.backend.clear_session()
        model = build_model()
        model.fit(x_pool[labeled], [y_known[i] for i in labeled], epochs=epochs, batch_size=32, verbose=0)
        acc = model.evaluate(x_test, y_test, batch_size=1024, verbose=0)[1]
        print(f"\n=== Ciclo {r + 1}/{n_rounds} | rotuladas: {len(labeled)} | acurácia atual: {acc:.1%} ===")

        unlabeled = np.setdiff1d(np.arange(POOL_SIZE), labeled)
        p = model.predict(x_pool[unlabeled], batch_size=1024, verbose=0)[:, 0]
        for i in unlabeled[np.argsort(-_entropy(p))[:k]]:
            print("\n" + decode(x_pool_raw[i])[:600] + "...")
            while (ans := input("Essa resenha é positiva (1) ou negativa (0)? ").strip()) not in ("0", "1"):
                pass
            y_known[i] = int(ans)
            labeled.append(i)
            human.append(i)

    concordou = np.mean([y_known[i] == y_pool[i] for i in human])
    print(f"\nVocê concordou com o rótulo oficial em {concordou:.0%} das {len(human)} resenhas que rotulou.")

# active_learning_com_humano()

## 9. Para ir além

- **Acurácia não é tudo:** em dados desbalanceados (fraude, doenças raras), active learning costuma ajudar ainda mais, pois acha os casos raros que o sorteio quase nunca traz.
- **Redundância no lote:** os top-K mais incertos podem ser resenhas quase iguais. Alternativas: *core-set*, *BADGE*, ou clustering dos embeddings para escolher um lote **diverso**.
- **Modelos maiores:** troque a CNN por um encoder pré-treinado (ex.: BERT via KerasHub). O active learning ajuda mais quando o modelo já tem boas representações.
- **Critério de parada:** em vez de um número fixo de ciclos, pare quando o ganho de acurácia por ciclo ficar abaixo do custo de rotular mais.
- **Mais de 2 classes:** a margem entre as duas classes mais prováveis (`margin`) costuma ser um ótimo score.

### Referências
- Gal, Islam & Ghahramani (2017), *Deep Bayesian Active Learning with Image Data*
- Settles (2009), *Active Learning Literature Survey*
- Exemplo oficial do Keras, *Review Classification using Active Learning*: https://keras.io/examples/nlp/active_learning_review_classification/